In [2]:
!uv pip install graphviz

Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Resolved 1 package in 161ms                                          
Prepared 1 package in 36ms                                               
Installed 1 package in 3ms                                  
 + graphviz==0.21


In [21]:
import re
import time


from colorama import Fore
from colorama import Style
from graphviz import Digraph

from dataclasses import dataclass

import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:

def completions_create(client, messages:list, model:str)->str:
    """
    Sends a request to the clients 'completion.create' method to interact with the language model.

    Args:
        client(OpenAI): The OpenAI client object
        messages(list[dict]): A List of messages object containing chat history for the model.
        model (str): The MOdel to use for generation tool calls and responses.

    Returns:-
        str: the contents of the models response
    """

    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt:str, role:str, tag:str="")->dict:
    """
    Builds a structured prompt that includes the role and content.

    Args:
        prompt(str): The actual content of the prompt.
        role (str): the role of the speaker (e.g:- user, assistant).

    Return:
    dict Adictonary representing the structured prompt.
    """

    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}

def update_chat_history(history:list,msg:str,role:str):
    """
    Updates the chat history by appending the latest response.

    Args:
        history(list): The List representing the current chat history.
        msg (str):The message to append.
        role(str): The role type (e.g. 'assistant', 'system')
    """

    history.append(build_prompt_structure(prompt=msg, role=role))



class ChatHistory(list):
    def __init__(self, messages:list|None=None, total_length:int=-1):
        """
        Initializes the queue with a fixed total length.

        Args:
            messages (list|None): Alist of initial messages
            total_length(int): The Maximum no of messages the chat history can hold.
        """

        if messages is None:
            messages = []

        super().__init__(messages)
        self.total_length = total_length


    def append(self, msg:str):
        """
        Add a message to the queue

        Args:
            msg(str): The message to be added to the queue
        """

        if len(self)==self.total_length:
            self.pop(0)
        super().append(msg)



class FixedFirstChatHistory(ChatHistory):
    def __init__(self, messages:list|None=None, total_length:int=-1):
        """Initialize the queue with a fixed total length.
    
            Args:
                messages(list|None): A list of Initial messages
                total_length (int): the maximum no of messages the chat history can hold.
        """
        super().__init__(messages, total_length)


    def append(self,msg:str):
        """Add a message to the queue. the first messages will always stay fixed.

            Args:
                msg(str): The message to be added to the queue
        """
        if len(self)==self.total_length:
            self.pop(1)
        super().aooend(msg)



def fancy_print(message:str)->None:
    """
    Display a fancy print message
    Args:
        message(str): The message to display.
        
    """

    print(Style.BRIGHT + Fore.CYAN +f"\n{'='*50}")
    print(Fore.MAGENTA + f"{message}")
    print(Style.BRIGHT + Fore.CYAN + f"{'='*50}\n")

    time.sleep(.5)


In [5]:
def fancy_step_tracker(step:int, total_step:int)->None:
    """
    Displays a fancy tracker for each iterations of the generation-reflection loop.

    Args:
        step(int): The currrent step in the loop.
        total_step(int): The total number of steps in the loop.
    """
    fancy_print(f"STEP {step+1}/{Total_step}")

In [6]:

@dataclass
class TagContentResult:
    """
    A data class to represent the result of the extracting tag content.

    Attributes:
        content(List[str]): A list of strings containing the content found between the specified tags.
        found(bool): A flag indicating wheather any content was found for the give tag.
    """

    content: list[str]
    found: bool



def extract_tag_content(text:str, tag:str)->TagContentResult:
    """
    Extract all content enclosed by specific tags (e.g., '<thought>', '<response>', etc)

    Parameters:
        text(str): The input string containing multiple potential tags.
        tag(str): The name of the tag to search for (e.g., 'thought', 'response').

    Returns:
        dict: A dictionary with the following keys:
            -'content' (list): A list of strings cintaining the content found between the specified tags.
            -'found' (bool): A flag indicating wheather any content was found for the given tag
    
    """
    # Build the regex pattern dynamically to find the multipple occurence of the tag
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"

    # use findall to capture all the content between the specified tag
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    # return  the dataclass instance with the result
    return TagContentResult(
        content = [content.strip() for content in matched_contents],
        found = bool(matched_contents)
    )



In [ ]:
# Crew

In [41]:

from collections import deque
class Crew:
    """
    A class representing a crew of agents working together.
    This class manages a group of agents, their dependencies and provides methods for running the agent in a topologically sorted order.

    Attributes:
        current_crew(Crew): Class-level variable to track the active Crew context
        agents(list): A list agents in the crew
    """
    current_crew = None

    def __init__(self):
        self.agents = []

    def __enter__(self):
        """
        Enters the context manager, setting this crew as the current active context

        Returns:
            Crew: The current crew instance
        """
        Crew.current_crew = self
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """
        Exits the context manager clearing the active context.

        Args:
            exc_type: the exception type, if an exception was raised
            exc_val: the exception value, if an exception was raised
            exc_tb: the traceback, if an exception was raised
        """

        Crew.current_crew = None

    def add_agent(self,agent):
        """
        Adds an agent to the crew.

        Args:
            agent: the agent to be added to the crew
        """
        self.agents.append(agent)

    @staticmethod
    def register_agent(agent):
        """
        Registers an agent with the current active crew context

        Args:
            agents: The agent to be registered.
        """

        if Crew.current_crew is not None:
            Crew.current_crew.add_agent(agent)


    def topological_sort(self):
        """
        Performs a topological sort of the agents based on their dependencies.

        Returns:
            list: A list of agents sorted in topological order.

        Raises:
            ValueError: if theres a circular dependency among agents.
        """

        in_degree = {agent: len(agent.dependencies) for agent in self.agents}
        queue = deque([agent for agent in self.agents if in_degree[agent]==0])

        sorted_agents = []
        while queue:
            current_agent = queue.popleft()
            sorted_agents.append(current_agent)

            for dependent in current_agent.dependents:
                in_degree[dependent]-=1

                if in_degree[dependent]==0:
                    queue.append(dependent)

        if len(sorted_agents)!=len(self.agents):
            raise ValueError(
                "Circular dependencies detected among agents, preventing a valid topological sort"
            )
        return sorted_agents


    def plot(self):
    
        """
        Plots the directed Acyclic Graph (DAG) of agents in the crew using GraphViz

        Returns:
            Digraph: A Graphviz Digraph object representing the agent dependencies.
        """

        dot = Digraph(format="png") #for inlne display

        for agent in self.agents:
            dot.node(agent.name)

            for dependency in agent.dependencies:
                dot.edge(dependency.name, agent.name)
        return dot


    def run(self):
        """
        Runs all the agents in the crew in topological sorted order

        This method executes each agents run methods and prints the result
        """

        sorted_agents = self.topological_sort()

        for agent in sorted_agents:
            fancy_print(f"RUNNING AGENT: {agent}")
            print(Fore.RED + f"{agent.run()}")

            









    

In [8]:
# Tool

In [9]:

import os
import json
import re
from dataclasses import dataclass
from typing import Callable
from openai import OpenAI

# from google.colab import userdata

def get_fn_signature(fn:Callable)->dict:
    """
    Generates the signature for a given function

    Args:
        fn(Callable): The function whose signature needs to be extracted.

    Returns:
        dict: A dictionary containing the function's name, description, and parameter types.
    """
    fn_signature = {
        "name": fn.__name__,
        "description": fn.__doc__,
        "parameters":{"properties":{}}
    }

    schema = {
        k:{"type":v.__name__} for k,v in fn.__annotations__.items() if k!='return'
    }

    fn_signature['parameters']['properties'] = schema
    return fn_signature


def validate_arguments(tool_call:dict, tool_signature:dict)->dict:
    """
    Validates and converts argument in the input dict to match the expected types.

    Args:
        tool_call(dict): A dict containing the arguments passed to the tool.
        tool_signature(dict): The expected function signature and parameter types.

    Returns:
        dict: The tool call dict with arguments converted to the correct types if necessary.
        
    """

    properties = tool_signature['parameters']['properties']

    # TODO: This is overly simplified but enough for simple tools

    type_mapping = {
        'int':int,
        'str':str,
        'bool':bool,
        'float':float
    }

    for arg_name, arg_value in tool_call['arguments'].items():
        expected_type = properties[arg_name].get('type')

        if not isinstance(arg_value, type_mapping[expected_type]):
            tool_call['arguments'][arg_name] = type_mapping[expected_type](arg_value)

    return tool_call


class Tool:
    """
    A class representing a tool that wraps a callable and its signature.

    Attributes:
        name(str): The name of the tool(function)
        fn(Callable): The function that the tool represents.
        fn_signature(str): json String representing the fn signature.
    """

    def __init__(self, name:str, fn:Callable, fn_signature:str):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        """
        Executes the tool(function) with provided arguments.

        Args:
            **kwargs: Keyword arguments passed to the function.

        Returns:
            The result of the function.
        """

        return self.fn(**kwargs)


def tool(fn:Callable):
    """
    A decorator that wraps a function into tool object.
    Args:
        fn(Callable): The function to be wrapped.

    Returns:
        Tool: A Tool object containing the function, its name, and signature.
        
    """

    def wrapper():
        fn_signature = get_fn_signature(fn)

        return Tool(
            name=fn_signature.get('name'), fn=fn, fn_signature=json.dumps(fn_signature)
        )
    return wrapper()

In [10]:

@dataclass
class TagContentResult:
    """
    A data class to represent the result of the extracting tag content.

    Attributes:
        content(List[str]): A list of strings containing the content found between the specified tags.
        found(bool): A flag indicating wheather any content was found for the give tag.
    """

    content: list[str]
    found: bool



def extract_tag_content(text:str, tag:str)->TagContentResult:
    """
    Extract all content enclosed by specific tags (e.g., '<thought>', '<response>', etc)

    Parameters:
        text(str): The input string containing multiple potential tags.
        tag(str): The name of the tag to search for (e.g., 'thought', 'response').

    Returns:
        dict: A dictionary with the following keys:
            -'content' (list): A list of strings cintaining the content found between the specified tags.
            -'found' (bool): A flag indicating wheather any content was found for the given tag

    """
    # Build the regex pattern dynamically to find the multipple occurence of the tag
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"

    # use findall to capture all the content between the specified tag
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    # return  the dataclass instance with the result
    return TagContentResult(
        content = [content.strip() for content in matched_contents],
        found = bool(matched_contents)
    )



In [22]:

BASE_SYSTEM_PROMPT = ""

REACT_SYSTEM_PROMPT = """
    You operate by running a loop with the following steps: Thought, Action, Observation.
    You are provided with function signature within <tools></tools> XML tags.
    You may call one or more functions to assist with the user query. Don't make assumptions about what value to plug
    into functions. Pay special attention to the properties 'types'. You should use those types as in a python dict.

    for each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follow

    <tool_call>
    {'name':<function-name>,'arguments':<args-dict>,'id':<monotonically-increasing-id>}
    </tool_call>

    Here are the available tools / actions:

    <tools>
    %s
    </tools>

    Example Session:

    <question> whats the current temprature in delhi</question>
    <thought>I need to get the current weather in delhi</thought>
    <tool_call>{"name": "get_current_weather","arguments":{"location": "delhi", "unit": "celsius"}, "id":0}</tool_call>

    You will be called again with this:
    <observation>{0:{"temperature":25, "unit": "celsius"}}</observation>

    You then output:
    <response>The Current temperature in delhi is 25 degree Celsius</response>

    Additional constraints:

    -If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> Tags.
    """



class ReactAgent:
    """
    A class that represents an agent uisng the ReAct logic that interacts with tools to process user inputs, make dicisions, and executes
     tool calls. the agent can run interactive sessions, collect tool signature, and process multiple tools calls in a given round of interaction.

     Attributes:
         client(OpenAI): The OpenAI client used to handle model-based completions.
         model(str): The name of the model used for generating responses. Default to 'GPT-4o'.
         tools(list[Tools]): A list of Tool instances available for execution.
         tools_dict: A dict mapping tool names to their corresponding Tool instances
    """

    def __init__(self, tools:Tool|list[Tool], model:str='gpt-4o', system_prompt:str=BASE_SYSTEM_PROMPT,api_key:str='')->None:
        
        self.client = OpenAI(api_key = os.environ.get('OPENAI_API_KEY'))
        self.model = model
        self.system_prompt = system_prompt

        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {tool.name: tool for tool in self.tools}

    def add_tool_signatures(self)->dict:
        """
        Collects the function signature of all available tools.

        Returns:
            str: A concatenated string of all tool function signature in JSON format.
        """
        return "".join([tool.fn_signature for tool in self.tools])


    def process_tool_calls(self, tool_calls_contents:list)->dict:
        """
        Processes each tool call validates arguments, executes the tools, and collects results.

        Args:
            tool_calls_content (list): List of strings each representing a tool call in json fromat.

        Returns:
            dict: A dictionary where the keys are tool call IDs and value are the results from the tools.
        """

        observations = {}
        for tool_call_str in tool_calls_contents:
            tool_call = json.loads(tool_call_str)
            tool_name = tool_call['name']
            tool = self.tools_dict[tool_name]

            print(Fore.GREEN +f"\nUsing Tool: {tool_name}")

            # Validate and  execute the tool call
            validated_tool_call = validate_arguments(
                tool_call, json.loads(tool.fn_signature)
            )

            print(Fore.GREEN + f"\nTool call dict: \n{validated_tool_call}")

            result = tool.run(**validated_tool_call['arguments'])
            print(Fore.GREEN + f"\nTool Result: \n{result}")

            # Store the result using the tool call ID
            observations[validated_tool_call['id']] = result

        return observations

    def run (self, user_msg:str, max_rounds:int=10)->str:
        """
        Executes a user interaction session where the agent processes user input, generates responses
        handles tool calls, and updates chat history until a final response is ready or the maximum number of rounds is reached.

        Args:
            user_msg(str): The users input message to start the interaction.
            max_rounds (int, optional): Maximum number of interaction rounds the agent should perform. default to 10.

        Returns:
            str: The final response generated by the agent after processing user input and any tool calls.
        """

        user_prompt = build_prompt_structure(
            prompt=user_msg, role='user', tag='question'
        )

        if self.tools:
            self.system_prompt+=("\n" + REACT_SYSTEM_PROMPT % self.add_tool_signatures())

        chat_history = ChatHistory([

            build_prompt_structure(
                prompt=self.system_prompt, role='system'
            ),
            user_prompt,
        ]
        )

        if self.tools:
        
            # Run the ReAct Loop for max_rounds

            for _ in range(max_rounds):
                completion = completions_create(self.client, chat_history, self.model)

                response = extract_tag_content(str(completion), "response")

                if response.found:
                    return response.content[0]

                thought = extract_tag_content(str(completion),'thought')
                tool_calls = extract_tag_content(str(completion),'tool_call')

                update_chat_history(chat_history, completion, 'assistant')

                print(Fore.MAGENTA + f"\nThought: {thought.content[0]}")


                if tool_calls.found:
                    observations = self.process_tools_calls(tool_calls.content)
                    print(Fore.BLUE + f"\bObservations: {observations}")
                    update_chat_history(chat_history, f"{observations}", "user")

        return completions_create(self.client, chat_history, self.model)
                    


In [23]:
from textwrap import dedent

class Agent:
    """
    Represents an AI agent that can work as part of a team to complete tasks.

    This class implements an agent with dependencies, context handling and task execution capabilitites.
    It can be used in a Multi-Agent system where agents colloborate to solve complex promplems.

    Attributes:
        name(str): The name of the agent.
        backstory(str): The backstory/background  of the agent.
        task_description(str): A description of the task assigned to the agent.
        task_expected_output(str): The expected format or content of the task output.
        react_agent(ReactAgent): An instance of ReactAgnet used for generating response.
        dependencies(list[Agents]): A list of Agent instances that this agent depends on.
        dependents(list[Agents]): A list of Agent instances that depends on this agents.
        context(str):Accumulated content information from other agents.

    Args:
        name(str): The name of the agent.
        backstory(str): The backstory/background  of the agent.
        task_description(str, ): A description of the task assigned to the agent.
        task_expected_output(str, optional): The expected format or content of the task output.
        tools(list[Tool]|None,optional): A list of Tool instances available to the agent. Defaults to None.
        llm(str,optional):The Name of the language model to use.Defaults to 'gpt-4o'.

    """

    def __init__(self,
                name:str,
                backstory:str,
                task_description:str,
                task_expected_output:str="",
                tools:list[Tool]|None=None,
                llm:str='gpt-4o'
                ):
        self.name = name
        self.backstory = backstory
        self.task_description = task_description
        self.task_expected_output = task_expected_output
        self.react_agent = ReactAgent(
            model=llm, system_prompt=self.backstory, tools = tools or []
        )

        self.dependencies:list[Agents] = []
        self.dependents:list[Agents] = []
        self.context = ""

        Crew.register_agent(self)

    def __repr__(self):
        return f"{self.name}"


    def __rshift__(self,other):
        """
        Defines the '>>' operator. this operator is used to indicate agent dependency.

        Args:
            other (Agent): the agent that depends on this agent
        """
        self.add_dependent(other)
        return other
    
    
    def __lshift__(self, other):

        """
        Defines the '<<' operator. this operator is used to indicate agent dependency in reverse.

        Args:
            other (Agent): the agent that depends on this agent
        """

         
        
        self.add_dependency(other)
        return other


    def __rrshift__(self, other):
        """
        Defines the '<<' operator. this operator is used to indicate agent dependency.

        Args:
            other (Agent): the agent that depends on this agent
        """

        self.add_dependency(other)
        return other

    
    def __rlshift__(self, other):
        """
        Defines the '<<' operator. when evaluated from right to left.
        this operator is used to indicate agent dependency in normal order
        

        Args:
            other (Agent): the agent that depends on this agent
        """
        self.add_dependent(other)
        return other


    def add_dependency(self, other):
        """
        Adds a dependency to this agent

        Args:
            other (Agent|list[Agent]): The agent(s) that this agent depends on.

        Raises:
            TypeError: If the dependency is not an agent or a list of Agents.
        """

        if isinstance(other, Agent):
            self.dependencies.append(other)
            other.dependents.append(self)

        elif isinstance(other,list) and all(isinstance(item,Agent) for item in other):
            for item in other:
                self.dependencies.append(item)
                item.dependents.append(self)
        else:
            raise TypeError("The dependency must be an instance or list of agents")

    def add_dependent(self, other):
        """
         Adds a dependent to this agent

        Args:
            other (Agent|list[Agent]): The agent(s) that this agent depends on.

        Raises:
            TypeError: If the dependent is not an agent or a list of Agents.
        
        """
        if isinstance(other, Agent):
            self.dependencies.append(other)
            other.dependents.append(self)

        elif isinstance(other,list) and all(isinstance(item,Agent) for item in other):
            for item in other:
                self.dependencies.append(item)
                item.dependents.append(self)
        else:
            raise TypeError("The dependency must be an instance or list of agents")

    def receive_context(self, input_data):
        """
        Recevies and stores context information from other agents.

        Args:
            input_data(str): The Context Information to be added
        """
        self.context+= f"{self.name} received context: \n{input_data}"

    def create_prompt(self):
        """
        Creates a prompt for the agent based on its task description, expected output, and context.

        Returns:
            str: The formatted prompt string.
        """

        prompt = dedent(
            f"""
            You are an AI agent. You are part of a team of agents working together to complete a task.
            I'm going to give you the task description enclosed in <task_description></task_description> tags. I'll also give
            you the available context from other agents in <context></context> tags. If the context is not available the <context></context> tags
            will be empty. You'll also receive the task expected output enclosed in <task_expected_output></task_expected_output> tags.
            With all this information you need to create the best possible response, always respecting the format as described in 
            <task_expected_output></task_expected_output> tags. If expected output is not available, just create a meaningful response to complete the task.

            <task_description>
            {self.task_description}
            </task_description>

            <task_expected_output>
            {self.task_expected_output}
            </task_expected_output>

            <context>
            {self.context}
            </context>

            Your Response:
            """
        
        ).strip()

        return prompt

    def run(self):
        """
        Runs the agents task and generates the output.
        This method creates a prompt, runs it through ReactAgent and passess the output to all dependent agents

        Returns:
            str: The output generated by agents.
        """

        msg = self.create_prompt()

        output = self.react_agent.run(user_msg=msg)

        for dependent in self.dependents:
            dependent.receive_context(output)

        return output
            
        

In [24]:
agent_example = Agent(
    name = "Poet Agent",
    backstory = "You are a well-known poet, who enjoys creating high quality poetry",
    task_description = "Write a poem about the meaning of life in less then 6 line",
    task_expected_output = "Just output the poem, without any title or introductory sentences"
)

In [27]:
print(agent_example.run())

In life’s embrace, a fleeting dance,  
Moments weave in chance's glance.  
Seek not to ask, nor quest to find,  
Meaning blooms in heart, not mind.  
Love's whispers echo through the strife,  
Eternal breath, the soul of life.  


In [28]:
agent_1 = Agent( name = "Poet Agent",
    backstory = "You are a well-known poet, who enjoys creating high quality poetry",
    task_description = "Write a poem about the meaning of life in less then 6 line",
    task_expected_output = "Just output the poem, without any title or introductory sentences"
)
agent_2 = Agent( name = "Poet Translator Agent ",
    backstory = "You are an expert translatorespecially skilled in ancient greek",
    task_description = "Translate a poem into ancient greek",
    task_expected_output = "Just output the translated  poem and nothing else"
)

In [29]:
agent_1 >> agent_2

Poet Translator Agent 

In [30]:
print("Agent 1 dependencies: ", agent_1.dependencies)
print("Agent 1 dependencies: ", agent_1.dependents)
print("Agent 2 dependencies: ", agent_2.dependencies)
print("Agent 2 dependencies: ", agent_2.dependencies)


Agent 1 dependencies:  [Poet Translator Agent ]
Agent 1 dependencies:  []
Agent 2 dependencies:  []
Agent 2 dependencies:  []


In [31]:
agent_1.run()

"In whispers of the night's embrace,  \nWe find our lives a fleeting trace,  \nA dance with time's relentless shore,  \nIn love and loss, we soar, explore,  \nTo seek, to feel, to simply be,  \nOur purpose threads eternity.  "

In [34]:
agent_2.run()

'<answer>ᾠδήν τινα ἐς τὴν ἀρχαίαν Ἑλληνικὴν γλῶσσαν μεθαρμόσας.</answer>'

In [42]:
with Crew() as crew:
    agent_1 = Agent( name = "Poet Agent",
    backstory = "You are a well-known poet, who enjoys creating high quality poetry",
    task_description = "Write a poem about the meaning of life in less then 6 line",
    task_expected_output = "Just output the poem, without any title or introductory sentences"
    )
    agent_2 = Agent( name = "Poet Translator Agent ",
        backstory = "You are an expert translatorespecially skilled in ancient greek",
        task_description = "Translate a poem into ancient greek",
        task_expected_output = "Just output the translated  poem and nothing else"
    )
    agent_3 = Agent( name = "Poet Translator Agent ",
        backstory = "You are an expert translatorespecially skilled in hindi",
        task_description = "Translate a poem into ancient greek",
        task_expected_output = "Just output the translated  poem and nothing else"
    )

    agent_1>>agent_2>>agent_3

In [43]:
# crew.plot()

In [44]:
crew.run()


RUNNING AGENT: Poet Translator Agent 

I'm sorry, I can only translate text to or from Hindi language.

RUNNING AGENT: Poet Translator Agent 

λυρέων ἀοιδὴν λαμπρὰ κλέος ἀθανάτοις,  
ὐμνος ἄμμον, φερὲ φῇ, ἀείδειν τε τέχναν,  
ὄλβιος εἰρήνης, θαυμαστὸν ἐν φῖλοις πόντῳ.

RUNNING AGENT: Poet Agent

In whispers of wind, life's secrets lie,  
A dance of stars ‘neath endless sky.  
Where gentle hearts in silence learn,  
To love, to dream, with each return.  
In fleeting breath, find purpose rife,  
For in each moment, blooms our life.  
